# (노트) Backpropagation

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [빅데이터분석]

### import 

In [1]:
import torch 

### 지난시간복습

`-` 회귀분석에서 손실함수에 대한 미분은 아래와 같은 과정으로 계산할 수 있다. 

- $loss=({\bf y}-{\bf X}{\bf W})^\top({\bf y}-{\bf X}{\bf W})={\bf y}^\top {\bf y} - {\bf y}^\top {\bf X}{\bf W} - {\bf W}^\top {\bf X}^\top {\bf y} + {\bf W}^\top {\bf X}^\top{\bf X}{\bf W} $

- $\frac{\partial}{\partial {\bf W}}loss=-2 {\bf X}^\top {\bf y} + 2{\bf X}^\top{\bf X}{\bf W} $

### 오늘수업의 목표

`-` 목표: 미분을 계산할 수 있는 2개의 컴퓨터(core)가 있다고 가정할 때, 위의 계산을 더 빠르게 할 방법이 있을까? 

- 체인룰 $\to$ 역전파기법 

`-` 어떻게 하면 미분계산을 여러 컴퓨터에게 적당히 나누어 수행시킬 수 있을까? 

### 체인룰 (어려운 하나의 미분을 손쉬운 여러개의 미분으로 나누는 기법)

`-` 손실함수는 아래와 같은 변환들을 거쳐서 계산되었다고 볼 수 있다. 

- ${\bf X} \to {\bf X}{\bf W} \to {\bf y}-{\bf X}{\bf W} \to ({\bf y}-{\bf X}{\bf W})^\top({\bf y}-{\bf X}{\bf W}) $

`-` 이 과정을 좀 더 세분화 하여 나타내면 아래와 같다. 

- $\bf u = X\color{red}{W}$, $\quad {\bf u}: n \times 1 $ 

- $\bf v = y - \color{red}{u}$, $\quad {\bf v}: n \times 1 $   

- $loss= \bf \color{red}{v}^\top \color{red}{v} $  

`-` 손실함수에 대한 미분은 아래와 같다. 

$$\frac{\partial}{\partial \bf W}loss = \frac{\partial}{\partial \bf W} \bf v^\top v$$

(그런데 이걸 어떻게 계산함?)

`-` 계산할 수 있는것들의 모음.. 

- $\frac{\partial }{\partial \bf v} loss = 2{\bf v}$  $\quad \to$ (n,1) 벡터

- $\frac{\partial}{\partial \bf u} {\bf v}^\top = - {\bf I}$ $\quad \to$ (n,n) 매트릭스 

- $\frac{\partial}{\partial \bf W} {\bf u}^\top = {\bf X}^\top$ $\quad \to$ (p,n) 매트릭스 

`-` 혹시.. 아래와 같이 쓸 수 있을까? 

$$\left(\frac{\partial }{\partial \bf W}{\bf u}^\top\right) 
\left(\frac{\partial }{\partial \bf u}{\bf v}^\top \right)
\left(\frac{\partial }{\partial \bf v}loss\right) 
=\frac{\partial {\bf u}^\top}{\partial \bf W}\frac{\partial {\bf v}^\top}{\partial \bf u} \frac{\partial loss}{\partial \bf v}$$ 

- 가능할것 같다. 뭐 기호야 정의하기 나름이니까! 

`-` 그렇다면 혹시 아래와 같이 쓸 수 있을까? 

$$\frac{\partial {\bf u}^\top}{\partial \bf W}\frac{\partial {\bf v}^\top}{\partial \bf u} \frac{\partial loss}{\partial \bf v}=\frac{\partial loss}{\partial \bf W}=\frac{\partial}{\partial \bf W} loss $$

- 이건 선을 넘는 것임. 

- 그런데 어떠한 공식에 의해서 가능함. 그 공식 이름이 체인룰이다. 

`-` 결국 정리하면 아래의 꼴이 되었다. 

$$\left(\frac{\partial }{\partial \bf W}{\bf u}^\top\right) 
\left(\frac{\partial }{\partial \bf u}{\bf v}^\top \right)
\left(\frac{\partial }{\partial \bf v}loss\right) =\frac{\partial}{\partial \bf W} loss $$

`-` 그렇다면? 

$$\left({\bf X}^\top\right)
\left(-{\bf I} \right) 
\left(2{\bf v}\right) 
=\frac{\partial}{\partial \bf W} loss $$

그런데 ${\bf v}= {\bf y}-{\bf u} = {\bf y}-{\bf X}{\bf W}$ 이므로 

$$ -2{\bf X}^\top ({\bf y} - {\bf X}{\bf W})= \frac{\partial}{\partial \bf W}loss$$ 

정리하면 

$$\frac{\partial}{\partial \bf W}loss= -2{\bf X}^\top{\bf y} + 2{\bf X}^\top{\bf X}{\bf W}$$ 

### 예시: 중간고사 문제..  

`-` 미분계수를 계산하는 문제였음 

`-` 체인룰을 이용하여 미분계수를 계산하여 보자. (미분 공식을 알고 있다는 전제)

In [2]:
ones= torch.ones(5)
x = torch.tensor([11.0,12.0,13.0,14.0,15.0])
X = torch.vstack([ones,x]).T
y = torch.tensor([17.7,18.5,21.2,23.6,24.2])

In [3]:
W = torch.tensor([3.0,3.0])

In [4]:
u = X@W 
v = y-u
loss = v.T @ v

`-` $\frac{\partial}{\partial \bf v} loss$ 의 계산 

In [5]:
X.T @ -torch.eye(5) @ (2*v) 

tensor([ 209.6000, 2748.6001])

- 참고로 중간고사 답은 

```python
X.T @ -torch.eye(5) @ (2*v) 
```

이므로 `tensor([ 41.9200, 549.7200])` 와 같이 나오면 되었습니다. 

#### 방법2: 미분공식을 모를 경우.

In [6]:
u,v,loss

(tensor([36., 39., 42., 45., 48.]),
 tensor([-18.3000, -20.5000, -20.8000, -21.4000, -23.8000]),
 tensor(2212.1799))

`-` $\frac{\partial}{\partial \bf v} loss$ 의 계산

In [7]:
_v = torch.tensor([-18.3000, -20.5000, -20.8000, -21.4000, -23.8000],requires_grad=True)

In [8]:
_loss = _v.T @ _v

In [9]:
_loss.backward()

In [10]:
_v.grad.data,_v.data

(tensor([-36.6000, -41.0000, -41.6000, -42.8000, -47.6000]),
 tensor([-18.3000, -20.5000, -20.8000, -21.4000, -23.8000]))

- 미분결과가 이론적인 값인 2v와 일치함 

`-` $ \frac{\partial }{\partial \bf u} {\bf v}^\top $ 의 계산 

In [11]:
_u = torch.tensor([36., 39., 42., 45., 48.],requires_grad=True)

In [12]:
_u

tensor([36., 39., 42., 45., 48.], requires_grad=True)

In [13]:
_v= y-_u ## 아까의 _v 와 또 다른 임시 v

In [14]:
_v.T.backward() 

RuntimeError: grad can be implicitly created only for scalar outputs

- 사실 토치에서 미분계산은 스칼라만 가능

그런데 $\frac{\partial }{\partial \bf u}{\bf v}^\top=\begin{bmatrix} \frac{\partial}{\partial \bf u} v_1 \\ \frac{\partial}{\partial \bf u} v_2 \\ ... \\  \frac{\partial}{\partial \bf u} v_n \end{bmatrix} $ 이므로 조금 귀찮은 과정을 거친다면 아래와 같은 알고리즘으로 계산할 수 있다. 

(0) $\frac{\partial}{\partial {\bf u}}{\bf v}^\top$의 결과를 기록할 매트릭스를 만든다. 적당히 `A`라고 만들자.

(1) `_u` 하나를 임시로 만들고 $v_1$을 미분하고 미분결과를 기록한다. 기록한 것을 `A`의 첫번재 행에 저장하자. 

(2) `_u`를 임시로 만들고 $v_2$를 미분한뒤 결과를 기록한다. 기록한 것을 `A`의 두번째 행에 저장한다. 

(3) (1)-(2)와 같은 작업을 $v_n$까지 반복한다. 

***(0)을 수행***

In [22]:
A = torch.zeros((5,5))
A

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

***(1)을 수행***

In [15]:
u,v

(tensor([36., 39., 42., 45., 48.]),
 tensor([-18.3000, -20.5000, -20.8000, -21.4000, -23.8000]))

In [17]:
_u = torch.tensor([36., 39., 42., 45., 48.],requires_grad=True) 
v1= (y-_u)[0] 

- 이때 $v_1 = g(f({\bf u}))$ 와 같이 표현할 수 있다. 여기에서 $f((u_1,\dots,u_n)^\top)=(y_1-u_1,\dots,y_n-u_n)^\top$, $g((v_1,\dots,v_n)^\top)=v_1$ 이라고 생각한다. 즉 $f$는 벡터 뺄셈을 수행하는 함수이고 $g$는 프로젝션 함수이다. $f: \mathbb{R}^n \to \mathbb{R}^n$인 함수이고 $g: \mathbb{R}^n \to \mathbb{R}$인 함수이다. 


In [18]:
v1.backward()

In [19]:
_u.grad.data

tensor([-1., -0., -0., -0., -0.])

In [25]:
A[0,:] = _u.grad.data

In [26]:
A

tensor([[-1., -0., -0., -0., -0.],
        [ 0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.]])

***(2)를 수행***

In [27]:
_u = torch.tensor([36., 39., 42., 45., 48.],requires_grad=True) 
v2= (y-_u)[1] 

In [28]:
v2.backward()

In [29]:
_u.grad.data

tensor([-0., -1., -0., -0., -0.])

In [30]:
A[1,:] = _u.grad.data

In [31]:
A

tensor([[-1., -0., -0., -0., -0.],
        [-0., -1., -0., -0., -0.],
        [ 0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.]])

***(3)을 수행 // 그냥 (1)~(2)도 새로수행하자..***

In [38]:
for i in range(5):
    _u = torch.tensor([36., 39., 42., 45., 48.],requires_grad=True) 
    _v= (y-_u)[i] 
    _v.backward()
    A[i,:] = _u.grad.data

In [39]:
A

tensor([[-1., -0., -0., -0., -0.],
        [-0., -1., -0., -0., -0.],
        [-0., -0., -1., -0., -0.],
        [-0., -0., -0., -1., -0.],
        [-0., -0., -0., -0., -1.]])

- 이론적인 결과인 $-{\bf I}$ 와 일치한다. 

`-` $\frac{\partial}{\partial \bf W} {\bf u}^\top$의 계산

$\frac{\partial}{\partial \bf W}{\bf u}^\top = \frac{\partial}{\partial \bf W}(u_1,\dots,u_n)= (\frac{\partial}{\partial \bf W}u_1,\dots,\frac{\partial}{\partial \bf W}u_n)$

In [64]:
B=torch.zeros((2,5))
B

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [65]:
_W = torch.tensor([3., 3.],requires_grad=True) 

In [66]:
(X@_W)

tensor([36., 39., 42., 45., 48.], grad_fn=<MvBackward>)

In [67]:
for i in range(5):
    _W = torch.tensor([3., 3.],requires_grad=True) 
    _u = (X@_W)[i]
    _u.backward()
    B[:,i] = _W.grad.data

In [69]:
B

tensor([[ 1.,  1.,  1.,  1.,  1.],
        [11., 12., 13., 14., 15.]])

In [70]:
X

tensor([[ 1., 11.],
        [ 1., 12.],
        [ 1., 13.],
        [ 1., 14.],
        [ 1., 15.]])

### 잠깐 생각해보자...

`-` 결국 위의 예제에 한정하여 임의의 ${\bf \hat{W}}$에 대하여 $\frac{\partial}{\partial {\bf \hat W}}loss$는 아래와 같이 계산할 수 있다. 
- (step1) $2{\bf v}$를 계산한다. 
- (step2) (step1)의 계산결과에서 $-{\bf I}$를 곱한다. 
- (step3) (step2)의 계산결과에서 ${\bf X}^\top$를 곱한다. 

`-` 다 알겠는데 step1에서 ${\bf v}$는 어떻게 구하나? 
- X $\to$ u=X@W $\to$ v=y-u 

- 이거 그런데 어차피 우리가 loss를 구하기 위해서 계산해야하는거 아니었어??... 
- step1: yhat, **step2: loss**, step3: derivative, step4: update

`-` **(중요)** step2에서 loss만 구할생각하지말고, 중간과정을 다 저장해라! (그중에 v가 있을테니까) 그리고 그걸 어떻게 적절하게 이용해보자.
- 각 층의 입력, 출력 다 텐서로 저장해라.

#### backpropation 알고리즘 모티브 

`-` 이제 아래와 같은 함수의 변환을 아키텍처로 이해하자. (함수의입력=레이어의입력, 함수의출력=레이어의출력)
- ${\bf X} \overset{l1}{\to} {\bf X}{\bf W} \overset{l2}{\to} {\bf y}-{\bf X}{\bf W} \overset{l3}{\to} ({\bf y}-{\bf X}{\bf W})^\top({\bf y}-{\bf X}{\bf W}) $

`-` 그런데 계산과정을 아래와 같이 요약할 수도 있다. (${\bf X} \to {\hat{\bf y}} \to loss$ 가 아니라 ${\bf W} \to loss({\bf W})$ 로 생각해봐요) 
- ${\bf W} \overset{l1}{\to} {\bf u} \overset{l2}{\to} {\bf v} \overset{l3}{\to} loss $

`-` 그렇다면 아래와 같은 사실을 관찰할 수 있다. 
- (step1)의 $2{\bf v}$는 function of ${\bf v}$이고, ${\bf v}$는 l3의 입력 (혹은 l2의 출력)
- (step2)의 $-{\bf I}$는 function of ${\bf u}$로 해석할 수 있고 ${\bf u}$는 l2의 입력이다. (혹은 l1의 출력) 
- 마찬가지 논리로 (step3)의 ${\bf X}^\top$는 function of ${\bf W}$라고 해석가능하다. 

`-` 이것은 사실 체인룰의 공식을 떠올리면 당연하다. 마지막 층은 미분과정은 $\frac{\partial}{\partial \bf v}loss$ 이었으니까 당연히 function of ${\bf v}$가 된다. 
- $y=x^2$라는 식이 성립할때 $\frac{dy}{dx}$는 당연히 $x$의 함수.. 

`-` 요약: $2{\bf v}$, $-{\bf I}$, ${\bf X}^\top$ 와 같은 핵심적인 값들이 사실 각 층의 입/출력 값들의 함수꼴로 표현가능하다 $\to$ 각 층의 입출력을 모두 기록하면 계산을 유리하게 할 수 있다
- 문득의문: 각 층의 입출력값 ${\bf v}, {\bf u}, {\bf W}$를 그로부터 $2{\bf v}$, $-{\bf I}$, ${\bf X}^\top$ 를 모른다면 말짱 헛수고 아닌가? 
- 의문해결: 어차피 우리가 쓰는 층은 선형 + (렐루, 시그모이드, ...) 정도가 전부임. 따라서 변환규칙은 미리 계산할 수 있음. 

`-` 결국 

(1) 순전파를 하면서 입출력값을 모두 저장하고 

(2) 그에 대응하는 층별 미분계수값 $2{\bf v}$, $-{\bf I}$, ${\bf X}^\top$을 구하고 

(3) 층별미분계수값을 역으로 곱하면 (그러니까 $2{\bf v} \times -{\bf I} \times {\bf X}^\top$ 를 수행, 그래야 차원이 맞으니까) 된다.

#### backpropagation 

(1) 순전파를 계산하고 각 층별 입출력값을 기록
- yhat=net(X)
- loss=loss_fn(yhat,y) 

(2) 층별 미분계수값을 계산 (함수의 종류는 정해져 있으므로 미분공식도 정해져 있음) 

(3) 역전파를 수행하여 손실함수의 미분값을 계산
- loss.backward()


`-` 참고로 (1)에서 층별 입출력값은 GPU의 메모리에 기록된다... 무려 GPU 메모리... 

`-` 작동원리를 GPU의 관점에서 요약 (슬기로운 GPU활용)

***gpu특징: 매트릭스 곱셈 전문가. (큰 차원의 매트릭스를 쉽게 곱함, 원리? 어머어마한 코어수)***
- 아키텍처 설정: 모형의 파라메터값을 GPU메모리에 올림. // `net.to("cuda:0")`
- 순전파 계산: ***중간계산결과를 모두 GPU메모리에 올림***. (순전파 계산만을 위해서라면 굳이 GPU에 있을 필요는 없으나 후에 역전파를 계산하기 위해 미리대비) // `net(X)`
- 오차 계산: `loss=loss_fn(yhat,y)`
- 역전파 계산: ***순전파 단계에서 계산한 결과를 활용하여*** 손실함수의 미분값을 계산한다. // `loss.backward()`
- 다음 순전파 계산: 이전의 값은 삭제하고 ***새로운 중간계산결과를 GPU메모리에 올림*** 
- 반복... 

### some comments 

`-` 역전파와 순전파는 방향때문에 생긴 용어이다. (거창한 의미는 없고 화살표가 반대처럼 보이니까) 

`-` 순전파는 yhat 혹은 loss를 계산하기 위해 사용한다. 보통은 yhat을 구하는 과정까지를 순전파라고 부르는것 같다. 

`-` 결국 순전파를 수행하고 (yhat을 구하고) 오차를 계산하고 다시 역전파를 계산하는 방식이다. (말이 너무 어려운데 책에 이렇게 다들 써놓음) 

`-` 이러한 이유로 오차를 구한후 역전파를 한다고 하여 오차역전파 기법이라고 불리기도 한다. 

`-` 오차 역전파 기법으로 학습하기 위해서는 GPU가 거의 필수적임 

`-` 순전파만을 위해서라면 GPU가 그다지 필요없다 $\to$ 따라서 학습된 모형을 이용하여 예측만 하려고 한다면 순전파만 필요한데 이때는 GPU가 필요없다. 

`-` 언제 GPU의 퍼포먼스가 CPU의 퍼포먼스를 능가하는가? 큰 사이즈의 매트릭스 연산이 많아질수록.. 

`-` 역전파를 수행하기 위해서는 반드시 그전에 순전파를 수행해야 한다. 

`-` 역전파기법은 체인룰 + $\alpha$ 이다. (역전파기법을 단순히 수학적인 체인룰이라고만 이해하는 사람은 역전파이전에 반드시 순전파를 선행해야 한다는 사실을 잘 이해하지 못한다.) 